<!-- colab-badge -->
[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RemusTeodorescu/DL-for-Engineers-Public-Course/blob/main/exercises/Ex07.1-poisson/Ex07.1_01_fdm_ground_truth.ipynb)

*Open this notebook in Google Colab. Its first code cell fetches the set's library files from the public course repository, so nothing needs uploading.*

<!-- course-header v3 -->

**Deep Learning for Engineering** · MSc, Aalborg University · 2026

Developed by **Remus Teodorescu** (ret@et.aau.dk), with support from Research Assistant **Noman Khan** (nomank@energy.aau.dk).

*Reference texts — read for the theory. Used as an **inspirational source** for this course, not as a source of its code:*

- Liu, *PINN with Python*, 2025.
- Raissi, Perdikaris & Karniadakis, *Physics-informed neural networks*, J. Comput. Phys. 378 (2019) 686–707.
- Prince, *Understanding Deep Learning*, MIT Press 2023.

Every notebook in this course has been **written and rewritten by the authors named above**. The code, the problems, the data and the exposition are **original to this course** and are not derived from any publisher's code listings or companion notebooks. Where notation matches a textbook's it is the standard notation of the field, and where an idea is a named author's it is cited as theirs in the text.

See `docs/PROVENANCE.md` for what each reference is cited for, set by set.

---

# Ex_07.1 · Notebook 01 — The Ground Truth, by Finite Differences

**Paired with L7.1 · Fundamentals of PINNs**

The slot from notebook 00, solved the classical way. Finite differences lay a
grid over the slot, replace every derivative by differences between
neighbouring nodes, and turn the equation into one linear system — which a
computer solves in milliseconds.

This notebook produces the answer the rest of the set is scored against, so it
has to earn that job. You show that the answer stops changing as the grid gets
finer, and then you make it as accurate as the method allows.

## What you will do

1. Write the heat source: the 32 wires' $RI^2$, with copper's resistance rising
   as it warms.
2. Solve the slot at rated current by finite differences, on a 161 × 321 grid,
   and find its hot spot.
3. Solve it again on coarser and finer grids, and show that 161 × 321 is the
   right mesh density for a ground truth.

## One thing to watch

A reference has to be far more accurate than what it judges. The networks in
notebook 02 get the hot spot to within about a tenth of a kelvin, so the ground
truth built here must be good to about a thousandth — a hundred times better.
Only then is the error you measure in notebook 02 the network's, and not the
reference's. Section 3 shows the grid that gives it.

---

## 0 · Setup

**What these cells do.** Fetch the library files, keep the results in your Google Drive so the later notebooks can read them, and import the modules.

**How.** The first cell downloads `course_core.py`, `pinn_core.py` and `problem.py` on Colab; the second asks for Drive access; the third imports them and checks they are really there.

In [ ]:
# files-cell v2 ----------------------------------------------------------
# This set's library files must sit beside the notebook. Locally they
# already do. On Google Colab, where a notebook opens on its own, they are
# fetched from the public course repository on every run, so a file left
# over from an earlier session cannot shadow it. Run this cell first.
import os, sys, urllib.request
FILES = ['course_core.py', 'pinn_core.py', 'problem.py']
URL = "https://raw.githubusercontent.com/RemusTeodorescu/DL-for-Engineers-Public-Course/main/exercises/Ex07.1-poisson/"
ON_COLAB = "google.colab" in sys.modules
for f in FILES:
    # on Colab always re-fetch: /content outlives a session and a stale copy
    # would silently shadow an updated one. Locally the repo's own files stay.
    if ON_COLAB or not os.path.exists(f):
        urllib.request.urlretrieve(URL + f, f)
        print("fetched", f)
# a module imported before this cell would keep the previous file's contents
for _m in ("course_core", "pinn_core", "problem"):
    sys.modules.pop(_m, None)
print("files ready:", ", ".join(FILES))

In [ ]:
# outputs-cell v1 --------------------------------------------------------
# Later notebooks in this set read results that earlier ones save. On Google
# Colab every notebook runs on its own temporary machine, so a file saved
# here is not there when the next notebook opens. This cell keeps the
# results in your Google Drive instead: approve the access request when it
# appears. If you decline it, or have no Google Drive, the results are
# downloaded to your computer when saved and the notebook that needs them
# asks for them back. Locally this cell does nothing.
import course_core as cc
cc.keep_outputs("Ex07.1_outputs")

In [ ]:
# --- setup: every Part 2 notebook opens with this cell ------------------
# Needs course_core.py, pinn_core.py and problem.py beside this notebook.
# On Colab the files cell above fetched them from the public course repository.
import os, time
for f in ("course_core.py", "pinn_core.py", "problem.py"):
    assert os.path.exists(f), f"{f} is missing - run the files cell above first"

from pinn_core import *                                  # noqa: F401,F403
import problem as pb
import numpy as np, torch, matplotlib.pyplot as plt

set_seed(88)  # fix the random start so a run repeats
print("device:", DEVICE, " dtype:", torch.get_default_dtype())

---

## 1 · The heat source

**What this cell does.** Defines $q(\theta, I)$, the heat the winding makes per cubic metre when the slot is $\theta$ kelvin above the wall and each wire carries $I$ amperes.

**How.** One metre of wire has resistance $\rho/A_{\text{wire}}$ and makes $RI^2$ of heat. Thirty-two of them, spread over the slot's cross-section, make

$$q = \text{fill}\cdot\rho(T)\,J^2, \qquad J = \frac{I}{A_{\text{wire}}}, \qquad \rho(T) = \rho_{20}\,[1 + \alpha\,(T - 20)], \qquad T = 90 + \theta$$

with `pb.FILL`, `pb.A_WIRE`, `pb.RHO_CU`, `pb.ALPHA_CU` and `pb.T_WALL` for the constants. The cell after it compares yours with `pb.heat_source`.

In [ ]:
# TODO 1: write the heat source q(theta, I), in W/m^3.
#
#   def heat_source(theta, I):
#       J = I / pb.A_WIRE                                                # A/m^2
#       rho = pb.RHO_CU * (1 + pb.ALPHA_CU * (pb.T_WALL + theta - 20))   # at T = 90 C + theta
#       return pb.FILL * rho * J ** 2                                    # 32 wires, over the slot
#
# The copper is at the wall temperature PLUS the rise. Leave out pb.T_WALL and
# its resistance is worked out at theta degrees - some ninety kelvin too cold.

raise NotImplementedError("Write heat_source(theta, I)")

In [ ]:
theta_test = np.array([0.0, 10.0, 40.0])        # kelvin above the wall
for I in (pb.I_RANGE[0], pb.I_RATED, pb.I_RANGE[1]):
    check(f"heat_source at {I:.1f} A", heat_source(theta_test, I) / pb.heat_source(theta_test, I),
          np.ones(3), tol=1e-12)

**What you should see.** Three `PASS` lines.

From here on the notebook uses `pb.heat_source`, the same function, so that
notebooks 02 and 03 compute exactly the heat this one does.

---

## 2 · The temperature field, by finite differences

**What this cell does.** Solves the slot at rated current on the grid this set uses as its ground truth, 161 × 321 nodes, finds the hot spot, and draws the mesh beside the temperature it produced.

**How.** The mesh has 161 nodes across the 10 mm width and $2 \times 161 - 1 = 321$ along the 20 mm depth, so the spacing is the same both ways, $h = 10/160 = 0.0625$ mm, and every cell is square. The nodes on the walls are $\theta = 0$ and are not unknowns; each of the $159 \times 319 = 50\,721$ nodes inside is one unknown. At every one of them the five-point stencil stands in for the Laplacian:

$$k_{\text{eff}}\,\frac{\theta_N + \theta_S + \theta_E + \theta_W - 4\theta_P}{h^2} + q(\theta_P) = 0,$$

using only the node's four neighbours, $h$ away. The heat is $q = c_0 + c_1\theta$, so every node gives one linear equation and the whole slot is one sparse system, $(k_{\text{eff}}L + c_1)\,\theta = -c_0$. `pb.fdm_matrix` builds it; `pb.fdm_solve` builds it and solves it. Why 161 nodes and not fewer or more is section 3.

In [ ]:
NX = 161                                                 # nodes across the slot; section 3 shows why
t0 = time.perf_counter()
X, Y, theta = pb.fdm_solve(pb.I_RATED, NX)               # 161 x 321 nodes, walls included
dt = time.perf_counter() - t0

i = np.unravel_index(theta.argmax(), theta.shape)
print(f"hot spot: {theta.max():.4f} K above the wall, {pb.T_WALL + theta.max():.1f} C, "
      f"at x = {X[i]*1e3:+.2f} mm, y = {Y[i]*1e3:+.2f} mm")
print(f"{(NX - 2) * (2 * NX - 3):,} unknowns, solved in {dt:.2f} s")

fig, axes = plt.subplots(1, 2, figsize=(10.5, 6.2))
pb.plot_fdm_mesh(NX, ax=axes[0])                         # the mesh, one corner enlarged
pb.plot_field(theta.ravel(), NX, 2 * NX - 1, ax=axes[1], wires=True,
              title="temperature rise at rated current")
plt.tight_layout(); plt.show()

**What you should see.** A hot spot of **18.74 K** at the centre of the slot —
108.7 °C in the copper — found in a fraction of a second, and a field
symmetric left to right and top to bottom: the heat is spread evenly and all
four walls are cooled alike.

**The left panel is the mesh the solve used.** At 0.0625 mm the nodes are too
close together to see on the whole slot, so the small box in its bottom-left
corner, 0.5 × 0.5 mm, is drawn again enlarged: the square nodes on the two
walls are known, every dot inside is an unknown, and one node is picked out
with the four neighbours its equation uses. Every interior node has the same
cross around it.

**The mesh does not follow the wires.** The winding is treated as one material
with its heat spread evenly (notebook 00), so a node inside a wire, on its edge
or between two wires gets exactly the same equation. The pale circles on the
left and the white ones on the right only show where the copper is; the field
does not see them.

---

## 3 · The right mesh density

**What this cell does.** Solves the same slot on six grids, from 11 to 321 nodes across, each twice as fine as the one before, times every solve, and solves once more on a 641 × 1281 grid to have something to measure them against.

**How.** For each grid it solves once to warm up, then three more times, and keeps the middle time — building the grid and the matrix included, because notebook 02 compares these times with a network's. The 641 grid takes about ten seconds; it is four times finer than the ground truth's, so its own error is sixteen times smaller, and it is used once, only as a yardstick.

In [ ]:
GRIDS = [11, 21, 41, 81, 161, 321]
sols, times = {}, {}
for nx in GRIDS:
    pb.fdm_solve(pb.I_RATED, nx)                           # warm-up, not timed
    ts = []
    for _ in range(3):
        t0 = time.perf_counter()
        _, _, th = pb.fdm_solve(pb.I_RATED, nx)            # grid + matrix + solve
        ts.append(time.perf_counter() - t0)
    sols[nx], times[nx] = th, float(np.median(ts))

_, _, ref = pb.fdm_solve(pb.I_RATED, 641)                  # the yardstick, 641 x 1281 nodes
print(f"yardstick: hot spot {ref.max():.5f} K on the 641 x 1281 grid")

**What you should see.** One line, the yardstick's hot spot, **18.74032 K**.

The next cell measures how far each grid is from it. The finer grid has every
node of the coarser ones: the 641 grid's nodes 0, $s$, $2s$, … sit exactly on
a grid of `nx` nodes when $s = 640/(nx - 1)$, so `ref[::s, ::s]` lines the two
up node for node.

In [ ]:
# TODO 2: the worst error of each grid, against the yardstick.
#
#   errs = {}
#   for nx in GRIDS:
#       s = (641 - 1) // (nx - 1)                             # the yardstick's nodes on this grid
#       errs[nx] = float(np.abs(sols[nx] - ref[::s, ::s]).max())   # worst error over the slot, K
#
# ref[::s, ::s] keeps every s-th node of the 641 grid: exactly the nodes that
# sit on the nx grid, so the two arrays line up.

raise NotImplementedError("Write the errs dictionary")

In [ ]:
TARGET = 1e-3                                             # K: a hundred times better than a network
for nx in GRIDS:
    mark = "  <- the ground truth" if nx == NX else ""
    print(f"  {nx:4d} x {2*nx - 1:4d} grid, {(nx-2)*(2*nx-3):6d} unknowns: worst error "
          f"{errs[nx]:.1e} K   {times[nx]*1e3:7.1f} ms{mark}")

h = np.array([2 * pb.A_HALF / (nx - 1) * 1e3 for nx in GRIDS])     # node spacing, mm
e = np.array([errs[nx] for nx in GRIDS])
t = np.array([times[nx] for nx in GRIDS]) * 1e3
fig, (a1, a2) = plt.subplots(1, 2, figsize=(11.0, 4.2))
a1.loglog(h, e, "o-", color=CYCLE[0], lw=2, label="finite differences")
a1.loglog(h, 2 * e[0] * (h / h[0]) ** 2, ":", color="0.4", label="slope 2, raised for comparison")
a2.loglog(t, e, "o-", color=CYCLE[0], lw=2)
for ax in (a1, a2):
    ax.axhline(TARGET, color=CYCLE[3], ls="--", lw=1.2)
    ax.text(ax.get_xlim()[0], TARGET * 1.3, " needed for a ground truth", color=CYCLE[3], fontsize=8)
    ax.set_ylabel("worst error  [K]")
for g, x1, x2, y in zip(GRIDS, h, t, e):
    a1.annotate(f" {g}", (x1, y), fontsize=8)
    a2.annotate(f" {g}", (x2, y), fontsize=8, color=CYCLE[1] if g == NX else "k")
a1.set_xlabel("node spacing $h$  [mm]"); a1.set_title("halving $h$ cuts the error four times")
a2.set_xlabel("time per solve  [ms]"); a2.set_title("and costs four to seven times the time")
k = GRIDS.index(NX)                                      # ring the ground truth
for ax, xv in ((a1, h), (a2, t)):
    ax.plot(xv[k], e[k], "o", ms=13, mfc="none", mec=CYCLE[1], mew=2)
a1.legend(frameon=False, loc="lower right"); plt.tight_layout(); plt.show()

**What you should see.** Errors from about 0.06 K on the 11-node grid down to
about $4 \times 10^{-5}$ K on the 321-node one, falling four times with each
halving of $h$ — the points lie on the slope-2 line, because the stencil's
error is proportional to $h^2$ — while the time grows four to seven times per
halving, a little faster than the number of unknowns, which grows four times.
The ground truth is ringed.

**Why 161 × 321.** A ground truth has to be about a hundred times more accurate
than what it judges, and the networks of notebook 02 are good to about a tenth
of a kelvin, so the target is $10^{-3}$ K, the dashed line. Read the table
against it:

* **41 × 81 and coarser** miss it. They are fine for an engineer who wants the
  hot spot to a hundredth of a kelvin, but not for a reference.
* **81 × 161** just reaches it, with no margin to spare.
* **161 × 321** is below it about five times over — two ten-thousandths of a
  kelvin, five hundred times better than a network — in about 0.2 s per
  current. This is the ground truth.
* **321 × 641** is four times better again, for six to seven times the time,
  an accuracy nothing in this set can use.

The times are your machine's; the ratios between them are what carry over.

---

## 4 · Save

**What this cell does.** Solves the slot on the ground-truth grid at six currents across the range and saves the fields, with every grid's error and time, for notebook 02 and the report.

**How.** Six calls of `pb.fdm_solve(I, NX)`, about a second in all. The fields go into `nb01_truth.npz` in `Ex07.1_outputs/`.

In [ ]:
TEST_I = np.array([10.0, 17.5, 25.0, pb.I_RATED, 38.0, 45.0])
fields = np.stack([pb.fdm_solve(I, NX)[2] for I in TEST_I])     # each 321 x 161

os.makedirs(cc.OUTPUT_DIR, exist_ok=True)
path = os.path.join(cc.OUTPUT_DIR, "nb01_truth.npz")
np.savez(path, currents=TEST_I, fields=fields, nx=NX,
         grids=np.array(GRIDS), times=np.array([times[n] for n in GRIDS]),
         errors=np.array([errs[n] for n in GRIDS]))
print("wrote", path, "-", fields.shape[0], "currents, hot spots",
      ", ".join(f"{f.max():.2f}" for f in fields), "K")
cc.saved(path)

**What you should see.** Six hot spots, from under 2 K at 10 A to about 40 K at
45 A, and the path of the saved file. The rise grows a little faster than
$I^2$ — hot copper makes more heat, as notebook 00 showed — so each current is
its own solve, not a rescaled copy of another.

---

## 5 · Before you move on

Answer these here. Each question builds part of an answer to one of the lecture's questions for the oral examination; the arrow under it says which, and the Questions slide at the end of the lecture has them in full.

1. Finite differences turned the slot into one linear system. What are its unknowns, how many did the 161 × 321 grid have, and what takes the place of the mesh in a physics-informed network?
   *→ L7.1 Q4*
2. Why did the error fall four times with each halving of $h$, and why is 161 × 321 the right grid for the ground truth rather than a coarser or a finer one?
   *→ L7.1 Q4*

*Write your answers here. You will copy them into the report in notebook 04, which adds them to what you submit.*

1.
2.

---

Next: **[notebook 02](https://colab.research.google.com/github/RemusTeodorescu/DL-for-Engineers-Public-Course/blob/main/exercises/Ex07.1-poisson/Ex07.1_02_pinn_soft_and_hard.ipynb)**, where a network learns the slot for every current at once, with its walls imposed softly and then hard, and is scored against this ground truth on accuracy and time.

<!-- two-forms -->
**Open notebook 02 as**

- **[the exercise](https://colab.research.google.com/github/RemusTeodorescu/DL-for-Engineers-Public-Course/blob/main/exercises/Ex07.1-poisson/Ex07.1_02_pinn_soft_and_hard.ipynb)**, where you write the missing lines, or
- **[the light version](https://colab.research.google.com/github/RemusTeodorescu/DL-for-Engineers-Public-Course/blob/main/exercises/Ex07.1-poisson/Ex07.1_02_pinn_soft_and_hard_light.ipynb)**, with every cell written out.